In [14]:
################ Import necessary libraries

%pip install pyblp
%pip install statsmodels
%pip install linearmodels
%pip install stargazer

import pandas as pd
import numpy as np
import pyblp 
import statsmodels.api as sm_api
from statsmodels.sandbox.regression.gmm import IV2SLS
from linearmodels import IV2SLS
from stargazer.stargazer import Stargazer
import matplotlib.pyplot as plt
import pickle

pyblp.options.digits = 2
pyblp.options.verbose = False
pyblp.__version__

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


'1.1.2'

In [88]:
################ Load data ready for nested logit
df = pd.read_csv('data_with_IV.csv')

# Generate nesting_id
df['nesting_ids'] = pd.factorize(df['fuel_type'])[0]

# Generate log of charging station stock
df['log_charging_IV'] = np.log(df['charging_stations_stock_lag']) 

# Create indicator for electric vehicles
df['is_electric'] = df['type'].apply(lambda x: 1 if x == '国产新能源乘用车' else 0)

# Make year an object variable
df['year'] = df['year'].astype(str)

# Rename columns to match pyblp requirements
df.rename(columns={
    'product_id': 'product_ids',
    'province_id': 'province_ids',
    'market_id': 'market_ids',
    'weighted_Avg_Price': 'prices',
    'market_share': 'shares'
}, inplace=True)

# calculate number of products in each market x nest
df['product_set_size'] = df.groupby(['market_ids', 'nesting_ids'])['product_ids'].transform('count')



In [89]:
################ Compute net prices of policies
# Import the province policy data
province_policy = pd.read_excel(r'C:\Users\Lenovo\Desktop\Dissertaion\China Data\Demand\Policy\Province_policy.xlsx')

# Import national policy data
national_policy = pd.read_excel(r'C:\Users\Lenovo\Desktop\Dissertaion\China Data\Demand\Policy\National_policy.xlsx')

# Sort province data
def expand_year_range(row):
    years = []
    if pd.notnull(row['year']):
        parts = str(row['year']).split('-')
        if len(parts) == 2:
            start, end = int(parts[0]), int(parts[1])
            years = list(range(start, end + 1))
        else:
            years = [int(parts[0])]
    return years

# Expand year
expanded_rows = []
for idx, row in province_policy.iterrows():
    for y in expand_year_range(row):
        new_row = row.copy()
        new_row['active_year'] = y
        expanded_rows.append(new_row)

province_policy_panel = pd.DataFrame(expanded_rows)

# Drop 'year' 
province_policy_panel.drop(columns=['year'], inplace=True)

# Rename columns to match the demand policy DataFrame
province_policy_panel.rename(columns={'active_year': 'year'}, inplace=True)
# Extract provinces and years
province_list = df['province'].unique()
years = list(range(2019, 2024))

# Construct policy panel
demand_policy_df = pd.DataFrame([(province, year) for province in province_list for year in years],
                        columns=['province', 'year'])

# Merge national policy data
demand_policy_df = demand_policy_df.merge(national_policy, on='year', how='left')

# Merge province policy data
demand_policy_df = demand_policy_df.merge(province_policy, on=['province', 'year'], how='left')

# Fill all NaN values with 0
demand_policy_df.fillna(0, inplace=True)

# Convert subsidy to units of 10000 CNY
demand_policy_df['sub'] = demand_policy_df['sub'] / 10000

# Sort year of demand policy DataFrame
demand_policy_df['year'] = demand_policy_df['year'].astype(int)
demand_policy_df = demand_policy_df.sort_values(['year', 'province']).reset_index(drop=True)
df['year'] = df['year'].astype(int)
df = df.sort_values(['year', 'province']).reset_index(drop=True)

# Merge the demand policy DataFrame with the main DataFrame
df = df.merge(demand_policy_df, on=['province', 'year'], how='left')

In [ ]:
# Compute net prices of policies
# Define the function to compute net prices
def compute_net_price(share):
    rg = share['range']
    p = share['prices']
    t = share['Tax']
    sub = share['sub']
    PHEV = share['sub_PHEV'] if share['fuel_type'] == 'PHEV' else 0
    BEV1 = share['sub_BEV_1'] if share['fuel_type'] == 'BEV' else 0
    BEV2 = share['sub_BEV_2'] if share['fuel_type'] == 'BEV' else 0
    BEV3 = share['sub_BEV_3'] if share['fuel_type'] == 'BEV' else 0

    def range_category(r, BEV1, BEV2, BEV3):
        if 250 <=r < 300: 
            return BEV1
        elif 300 <= r < 400:
            return BEV2
        elif 400 <= r:
            return BEV3
        else:
            return 0
    BEV = range_category(rg, BEV1, BEV2, BEV3)
    return p * t - sub - PHEV - BEV

# Compute net prices
df['net_prices'] = df.apply(compute_net_price, axis=1)

In [ ]:
# Define list of IVs
iv_list = [
    'cost_shifter', 
    'product_set_size',
    'euclidean_range',
    'local_range',
    'euclidean_battery',
    'local_battery',
    'euclidean_power',
    'local_power'
]

iv_str = ' + '.join(iv_list)


In [ ]:
# Check for variations in number of available products over time and across provinces

# 统计每个省份每年可用产品数
product_counts = df.groupby(['province_ids', 'year'])['product_ids'].nunique().reset_index(name='num_products')
print(product_counts)

# 检查每年不同省份的产品数分布
print("\n产品数在每年各省的分布：")
print(product_counts.groupby('year')['num_products'].describe())

# 检查每个省份不同年份的产品数分布
print("\n产品数在各省每年的分布：")
print(product_counts.groupby('province_ids')['num_products'].describe())

In [96]:
################ 2SLS model using custom instruments

# 计算 log(sjm), log(s0m), log(sj/g)
df['log_sjm'] = np.log(df['shares'])
df['log_s0m'] = np.log(1 - df.groupby('market_ids')['shares'].transform('sum'))
df['log_sj_g'] = np.log(df['shares'] / df.groupby(['market_ids', 'nesting_ids'])['shares'].transform('sum'))
df['log_charging_stock'] = np.log(df['charging_stations_stock'])
df['Intercept'] = 1


# Define the formula for the IV2SLS model
formula = f'''
(log_sjm - log_s0m) ~ 0 + is_electric*log_charging_IV + range + power + battery_capacity
    + [net_prices + log_sj_g ~ {iv_str}]
'''

iv_model = IV2SLS.from_formula(formula, data=df).fit(cov_type="clustered", clusters=df['market_ids'])

# 
main_vars = ['Intercept', 'net_prices', 'power', 'range', 'battery_capacity', 'log_charging_IV', 'log_sj_g']

print(iv_model.summary)

                          IV-2SLS Estimation Summary                          
Dep. Variable:                log_sjm   R-squared:                      0.9877
Estimator:                    IV-2SLS   Adj. R-squared:                 0.9877
No. Observations:               33545   F-statistic:                 7.568e+04
Date:                Mon, Jul 28 2025   P-value (F-stat)                0.0000
Time:                        22:18:55   Distribution:                  chi2(8)
Cov. Estimator:             clustered                                         
                                                                              
                                      Parameter Estimates                                      
                             Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
-----------------------------------------------------------------------------------------------
is_electric                    -12.799     0.6552    -19.536     0.0000     -14.

In [97]:
# Check first stage results
print(iv_model.first_stage.summary)

             First Stage Estimation Results            
                                net_prices     log_sj_g
-------------------------------------------------------
R-squared                           0.8392       0.8950
Partial R-squared                   0.1026       0.0559
Shea's R-squared                    0.0994       0.0542
Partial F-statistic                 591.47       282.72
P-value (Partial F-stat)            0.0000       0.0000
Partial F-stat Distn               chi2(8)      chi2(8)
============================= ============ ============
is_electric                        -13.585      -2.4318
                                 (-8.4120)    (-6.7263)
log_charging_IV                    -0.8260      -0.3388
                                 (-8.5446)    (-13.787)
range                               0.0083      -0.0021
                                  (13.188)    (-11.358)
power                               0.1156       0.0052
                                  (37.871)     (

In [23]:
################ Load supply-side data
supply_df = pd.read_csv('supply_side_data.csv')
charging_policy_df = pd.read_excel(r"C:\Users\Lenovo\Desktop\Dissertaion\China Data\Charging\Charging_policy.xlsx")

In [25]:
################ Process charging policy data and merge with supply-side data
def expand_year_range(row):
    years = []
    if pd.notnull(row['year']):
        parts = str(row['year']).split('-')
        if len(parts) == 2:
            start, end = int(parts[0]), int(parts[1])
            years = list(range(start, end + 1))
        else:
            years = [int(parts[0])]
    return years

# 展开年份区间
expanded_rows = []
for idx, row in charging_policy_df.iterrows():
    for y in expand_year_range(row):
        new_row = row.copy()
        new_row['active_year'] = y
        expanded_rows.append(new_row)

charging_policy_df = pd.DataFrame(expanded_rows)

charging_policy_df = charging_policy_df[['province', 'active_year', 'sub_fix', 'sub_ope']]

# Rename active_year to year for consistency
charging_policy_df.rename(columns={'active_year': 'year'}, inplace=True)

# Merge charging policy data with supply-side data
supply_df = supply_df.merge(charging_policy_df, on=['province', 'year'], how='left')
supply_df['sub_fix'] = supply_df['sub_fix'].fillna(0)
supply_df['sub_ope'] = supply_df['sub_ope'].fillna(0)

In [29]:
################ Supply side model

# Create a time trend variable
supply_df['time_trend'] = supply_df['year'].astype(int) - supply_df['year'].astype(int).min() + 1

# Add time_trend
formula = 'log(charging_stations_stock) ~ [log(EV_stock) ~ road_fuel_IV + num_models_in_market + sales_weighted_avg_range] + sub_fix + sub_ope + C(province) + time_trend'

supply_model = IV2SLS.from_formula(formula, data=supply_df).fit(
)
print(supply_model.summary)
print(supply_model.first_stage.summary)

                               IV-2SLS Estimation Summary                               
Dep. Variable:     log(charging_stations_stock)   R-squared:                      0.9737
Estimator:                              IV-2SLS   Adj. R-squared:                 0.9662
No. Observations:                           155   F-statistic:                 1.053e+06
Date:                          Mon, Jul 28 2025   P-value (F-stat)                0.0000
Time:                                  17:40:19   Distribution:                 chi2(35)
Cov. Estimator:                          robust                                         
                                                                                        
                                   Parameter Estimates                                   
                       Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
-----------------------------------------------------------------------------------------
sub_fix           

In [ ]:
# Export model summary as LaTeX
# Stargazer does not support different covariate orders for each model,
# so the best practice is to include all you want to show and accept blanks for the other.
main_vars = ['Intercept', 'log(EV_stock)', 'time_trend', 'shift_share_IV']

stargazer = Stargazer([supply_model])
stargazer.covariate_order(main_vars)

with open('supply_model_stargazer.tex', 'w', encoding='utf-8') as f:
    f.write(stargazer.render_latex())


In [15]:
# 保存需求侧模型结果
with open('demand_model_results.pkl', 'wb') as f:
    pickle.dump(iv_model, f)

# 保存供给侧模型结果
with open('charging_station_model_results.pkl', 'wb') as f:
    pickle.dump(supply_model, f)